# CFPB data inspection

## Business context

The [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/) holds consumer complaints and is published by the Consumer Financial Protection Bureau (CFPB). Each record keeps the `Product` and `Issue` options the consumer chose, according to the form in force when the complaint was submitted. Some complaints also carry a free-text narrative: 3,836,659 of the 17,094,898 rows in the snapshot used (22.4%).

A complaint goes through a few steps, described by the CFPB itself in its [complaint process](https://www.consumerfinance.gov/complaint/process/). The consumer submits it online or by phone, or another government agency forwards it to the CFPB, which sends it to the company named in it; when another agency is better able to help, the complaint goes to that agency instead. The company generally responds within 15 days and, in some cases, gives a final response within 60 days, and the consumer has 60 days to review that response. The complaint is published in the database after the company responds or after 15 days, whichever comes first, without names, contact details, account numbers or other information that directly identifies the consumer ([how the CFPB shares complaint data](https://www.consumerfinance.gov/complaint/data-use/)). On August 14, 2026, the CFPB [stopped publishing narratives](https://www.consumerfinance.gov/about-us/newsroom/the-cfpb-to-cease-discretionary-publication-of-complaint-narratives-and-visualizations/) and moved them to its FOIA Reading Room, which limits exact reconstruction of the narrative snapshot used here.

Routing a narrative to the right product family is the operational task of the project, which works with nine families. The critical class, `debt_credit_management`, has 5,437 rows among 3.8 million narratives (0.14%). Routing failures are expensive in that class, and every classical approach in the project failed on it.

Two readings are refused. Complaint volume does not measure market prevalence or consumer harm, because the target is routing to historical form categories. And `Issue` is not a stable global label: the [field reference](https://cfpb.github.io/api/ccdb/fields.html) states that its possible values depend on `Product`, 178 raw labels collide across families, and the form options changed over time, as in the [August 2023 product and issue options](https://files.consumerfinance.gov/f/documents/cfpb_consumer_complaint_form_product_issue_options_August_2023_FINAL.pdf).

This notebook performs the initial inspection of the data behind these choices.

Thin entry point for the initial profile. The executed notebook 01 at the root remains as a historical record; this version calls `src`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'dataset' / 'processed' / 'complaints.parquet').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not find project root')
    PROJECT_ROOT = parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from consumer_complaint_intelligence.config import ProjectPaths
from consumer_complaint_intelligence.data import read_parquet_sample

paths = ProjectPaths.from_root(PROJECT_ROOT)
columns = [
    'Date received',
    'Product',
    'Issue',
    'Consumer complaint narrative',
]
sample = read_parquet_sample(
    paths.require_parquet(), limit=25_000, columns=columns
)
sample.head(10)


The sample is limited to 25,000 rows and is not representative of the corpus. Taxonomy and split decisions belong to S0.
